# microWakeWord — Train any wake word

A single-notebook trainer for custom wake words on ESPHome `micro_wake_word` devices
(M5Stack Atom Echo, Voice PE, etc).

Two modes:
- **`generate`** — Piper TTS generates ~30k positive samples + your confusables in Colab.
  Easiest path. Requires an IPA pronunciation of your wake word.
- **`bundle`** — You upload a zip with your own samples (real recordings, ElevenLabs
  voices, accent-matched TTS). Higher quality but you do the prep work.

## Required runtime
**Runtime → Change runtime type → A100 GPU + High-RAM**.
T4 OOMs during validation. A100 + High-RAM gives 40 GB VRAM + 85 GB system RAM.

## What you get
A `<output_name>.tflite` (~60 KB) + companion `.json` manifest, ready to drop into
your ESPHome config under `micro_wake_word: models:`.

## Workflow
1. Edit the **CONFIGURE HERE** cell below for your wake word
2. **Runtime → Run all**, walk away ~45 minutes
3. Find `<output_name>.tflite` + `.json` in your Drive folder when it's done
4. Test on hardware — likely needs manifest tuning (cutoff, sliding_window) for
   your specific model. See the deployment notes at the end.

Built from working production deployment of "Hey Harold" — all known
upstream bugs are patched in this notebook.


In [1]:
# ╔═══════════════════════════════════════════════════════════════════════╗
# ║                    CONFIGURE YOUR WAKE WORD HERE                       ║
# ╚═══════════════════════════════════════════════════════════════════════╝

# ─── Wake word identity ───
WAKE_WORD = "Okay PRAT"             # Human-readable name (shown in HA)
OUTPUT_NAME = "okay_prat"           # Filename (no spaces, lowercase)
AUTHOR = "prat"                     # Goes in the manifest
AUTHOR_WEBSITE = "https://github.com/prat"

# ─── Mode ───
MODE = "generate"   # "generate" (Piper makes samples) | "bundle" (you uploaded a zip)

# ─── Output folder (created if missing). On Kaggle this is under /kaggle/working
#     so it survives as kernel Output; on Colab it would be a Drive folder. ───
DRIVE_FOLDER = "wakeword_training_okay_prat"

# ─── If MODE == "generate" ───
# Fallback IPA (manually transcribed, since espeak-ng isn't available in this
# sandbox to verify). A later cell auto-recomputes this via espeak-ng once
# it's installed, so this value is just a fallback if that step fails.
WAKE_WORD_IPA_US = "oʊkˈeɪ prˈæt"   # US English pronunciation (approx, auto-verified below)
WAKE_WORD_IPA_UK = "əʊkˈeɪ prˈæt"   # UK English (set to None to skip second pass)
SAMPLES_US = 30000                  # ~12 min on T4, ~7 min on A100
SAMPLES_UK = 15000                  # 0 to skip UK pass

# Confusable phrases — should NOT trigger your wake word.
# Include: phonetic neighbors, the bare name without prefix, common
# false-trigger phrases like other assistant names.
CONFUSABLE_PHRASES = [
    # General "okay X" near-misses
    "okay then", "okay now", "okay so", "okay you", "okay wait",
    # Other assistant wake words (must not steal yours)
    "hey siri", "hey google", "okay google", "hey alexa", "okay nabu", "hey jarvis",
    # WAKE-WORD SPECIFIC — phonetic neighbors of "prat"
    "okay pat", "okay brat", "okay rat", "okay prad", "okay plat", "okay pratt",
    "prat", "a prat", "prattle",                              # bare name + similar
]
SAMPLES_PER_CONFUSABLE = 1000   # 500 = light, 1000 = recommended, 2000 = max

# ─── If MODE == "bundle" ───
# On Kaggle, upload your bundle as a private Kaggle Dataset and attach it to
# this kernel; it will appear under /kaggle/input/<dataset-slug>/data_bundle.zip
BUNDLE_NAME = "data_bundle.zip"

# ─── Manifest tuning (works well as defaults; tune after on-device testing) ───
PROBABILITY_CUTOFF = 0.85          # 0.5 = too lenient, 0.97 = okay_nabu strict
SLIDING_WINDOW_SIZE = 5            # 5 = okay_nabu default; higher = slower fire
TENSOR_ARENA_SIZE = 50000          # 30000 works, 50000 has more headroom

# ─── Languages this model is trained for (manifest metadata) ───
TRAINED_LANGUAGES = ["en"]         # add "it", "es" etc if you have multi-accent samples

# ╔═══════════════════════════════════════════════════════════════════════╗
# ║                           END OF CONFIG                                ║
# ╚═══════════════════════════════════════════════════════════════════════╝
print(f"Training '{WAKE_WORD}' as {OUTPUT_NAME} in mode={MODE!r}")
print(f"Output -> /kaggle/working/{DRIVE_FOLDER}/{OUTPUT_NAME}.tflite")


Training 'Okay PRAT' as okay_prat in mode='generate'
Output -> /kaggle/working/wakeword_training_okay_prat/okay_prat.tflite


In [2]:
# === Set up working dirs (Kaggle equivalent of the Colab Drive-mount cell) ===
import os

# /content is used as scratch throughout this notebook (large intermediate
# data: raw samples, negative datasets, spectrogram feature mmaps). It is NOT
# under /kaggle/working, so none of that bulk gets swept into kernel Output --
# only the final .tflite/.json (copied to DRIVE_DIR below) do.
os.makedirs('/content', exist_ok=True)
os.chdir('/content')  # match Colab's default cwd, which the rest of the notebook assumes

DRIVE_DIR = f'/kaggle/working/{DRIVE_FOLDER}'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Output folder: {DRIVE_DIR}')

if MODE == 'bundle':
    # Expects a Kaggle Dataset attached to this kernel containing data_bundle.zip
    import glob
    candidates = glob.glob(f'/kaggle/input/*/{BUNDLE_NAME}')
    assert candidates, (
        f'MODE=bundle but no {BUNDLE_NAME} found under /kaggle/input/*/. '
        'Attach a Kaggle Dataset containing it to this kernel.')
    BUNDLE_PATH = candidates[0]
    print(f'Found bundle: {BUNDLE_PATH} ({os.path.getsize(BUNDLE_PATH)/1024/1024:.1f} MB)')


Output folder: /kaggle/working/wakeword_training_okay_prat


In [3]:
# === GPU sanity check ===
import subprocess, sys
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print('TensorFlow sees GPUs:', gpus)
assert gpus, 'No GPU visible to TensorFlow -- check Settings > Accelerator on this Kaggle kernel.'


Wed Sep 23 14:00:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
# === Install microWakeWord (kernel-restart-free) ===
# Workarounds for two upstream bugs:
#  1. kahrendt/microWakeWord setup.py has no find_packages() — non-editable
#     install skips the audio/ subpackage. Editable install needs kernel
#     restart, breaks Run All. Fix: install deps + sys.path.insert().
#  2. train.py calls .numpy() on values that newer TF returns as numpy
#     arrays already. Patch with hasattr() guard.
import os, sys, subprocess, importlib, re

DEPS = [
    'audiomentations', 'audio_metadata', 'datasets', 'mmap_ninja', 'numpy',
    'pymicro-features', 'pyyaml', 'tensorflow>=2.16', 'webrtcvad-wheels',
    'ai-edge-litert',
    'git+https://github.com/whatsnowplaying/audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f',
]
print('Installing dependencies...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + DEPS, check=True)

if not os.path.exists('microWakeWord'):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/kahrendt/microWakeWord'], check=True)

MWW_DIR = '/content/microWakeWord'
if MWW_DIR not in sys.path:
    sys.path.insert(0, MWW_DIR)
importlib.invalidate_caches()

fp = '/content/microWakeWord/microwakeword/train.py'
src = open(fp).read()
patched = re.sub(
    r'(\b[a-zA-Z_]+\["[a-z]+"\])\.numpy\(\)',
    r'(\1.numpy() if hasattr(\1, "numpy") else \1)',
    src
)
n = patched.count('hasattr') - src.count('hasattr')
if n > 0:
    open(fp, 'w').write(patched)
    print(f'Patched {n} .numpy() calls in train.py')

import microwakeword
from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration
print('OK: microwakeword.audio.* imports clean')


Installing dependencies...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.1/86.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.6/119.6 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.9/82.9 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.3/21.3 MB 75.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.1/82.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.2/52.2 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.5/248.5 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 kB 5.4 MB/s eta 0:00:00


Cloning into 'microWakeWord'...


OK: microwakeword.audio.* imports clean


In [5]:
# === Data preparation ===
# Mode-aware: either unzip user's bundle, or generate samples inline via Piper.
import os, zipfile

os.chdir('/content')

if MODE == 'bundle':
    print(f'Extracting {BUNDLE_PATH}...')
    with zipfile.ZipFile(BUNDLE_PATH, 'r') as zf:
        zf.extractall('/content')
    for d in ['generated_samples', 'real_recordings', 'confusable_negatives']:
        p = f'/content/{d}'
        if os.path.exists(p):
            n = sum(1 for _ in os.scandir(p) if _.name.endswith('.wav'))
            print(f'  {d}: {n} WAVs')
        else:
            print(f'  {d}: MISSING (will train without)')

elif MODE == 'generate':
    # Defer to the Piper sample-gen cells below
    print('MODE=generate — Piper sample-gen cells will produce samples')

else:
    raise ValueError(f'Unknown MODE: {MODE!r}. Use "bundle" or "generate".')


MODE=generate — Piper sample-gen cells will produce samples


In [6]:
# === Piper sample generator install (skipped if MODE=bundle) ===
if MODE == 'generate':
    import glob, os, shutil, subprocess, sys, urllib.request
    PIPER_REPO_DIR = '/content/piper'
    PIPER_SAMPLE_GENERATOR_DIR = '/content/piper-sample-generator'

    subprocess.run(['apt-get', '-qq', 'install', '-y', 'espeak-ng'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
                    'pip', 'setuptools', 'wheel', 'cython'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
                    'piper-tts', 'piper-sample-generator'], check=True)

    if not os.path.exists(PIPER_REPO_DIR):
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/rhasspy/piper', PIPER_REPO_DIR], check=True)
    if not os.path.exists(PIPER_SAMPLE_GENERATOR_DIR):
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/rhasspy/piper-sample-generator',
                        PIPER_SAMPLE_GENERATOR_DIR], check=True)

    PIPER_PYTHON_DIR = f'{PIPER_REPO_DIR}/src/python'
    MA_DIR = f'{PIPER_PYTHON_DIR}/piper_train/vits/monotonic_align'
    MA_IMPORT_DIR = f'{MA_DIR}/monotonic_align'
    MA_BUILD_DIR = f'{MA_DIR}/piper_train/vits/monotonic_align'

    shutil.rmtree(f'{PIPER_PYTHON_DIR}/build', ignore_errors=True)
    shutil.rmtree(MA_IMPORT_DIR, ignore_errors=True)
    shutil.rmtree(f'{MA_DIR}/piper_train', ignore_errors=True)
    os.makedirs(MA_IMPORT_DIR, exist_ok=True)
    os.makedirs(MA_BUILD_DIR, exist_ok=True)
    open(f'{MA_IMPORT_DIR}/__init__.py', 'a').close()
    subprocess.run(f'cd {MA_DIR} && {sys.executable} setup.py build_ext --inplace',
                   shell=True, check=True)
    built = next(iter(glob.glob(f'{MA_BUILD_DIR}/core.*')), None)
    assert built, 'monotonic_align core extension build failed'
    shutil.copy2(built, MA_IMPORT_DIR)

    for path in (PIPER_PYTHON_DIR, PIPER_SAMPLE_GENERATOR_DIR):
        if path not in sys.path:
            sys.path.insert(0, path)

    MODEL_PATH = 'models/en_US-libritts_r-medium.pt'
    MODEL_CONFIG_PATH = f'{MODEL_PATH}.json'
    os.makedirs('models', exist_ok=True)
    if not os.path.exists(MODEL_PATH):
        print('Downloading libritts_r model (~75 MB)...')
        urllib.request.urlretrieve(
            'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt',
            MODEL_PATH)
        urllib.request.urlretrieve(
            'https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/libritts_r/medium/en_US-libritts_r-medium.onnx.json',
            MODEL_CONFIG_PATH)
    print('Piper ready')
else:
    print('Skipped (MODE != generate)')


Selecting previously unselected package libpcaudio0:amd64.
(Reading database ... 125186 files and directories currently installed.)
Preparing to unpack .../libpcaudio0_1.1-6build2_amd64.deb ...
Unpacking libpcaudio0:amd64 (1.1-6build2) ...
Selecting previously unselected package libsonic0:amd64.
Preparing to unpack .../libsonic0_0.2.0-11build1_amd64.deb ...
Unpacking libsonic0:amd64 (0.2.0-11build1) ...
Selecting previously unselected package espeak-ng-data:amd64.
Preparing to unpack .../espeak-ng-data_1.50+dfsg-10ubuntu0.1_amd64.deb ...
Unpacking espeak-ng-data:amd64 (1.50+dfsg-10ubuntu0.1) ...
Selecting previously unselected package libespeak-ng1:amd64.
Preparing to unpack .../libespeak-ng1_1.50+dfsg-10ubuntu0.1_amd64.deb ...
Unpacking libespeak-ng1:amd64 (1.50+dfsg-10ubuntu0.1) ...
Selecting previously unselected package espeak-ng.
Preparing to unpack .../espeak-ng_1.50+dfsg-10ubuntu0.1_amd64.deb ...
Unpacking espeak-ng (1.50+dfsg-10ubuntu0.1) ...
Setting up libpcaudio0:amd64 (1.1-6

Cloning into '/content/piper'...
Cloning into '/content/piper-sample-generator'...
performance hint: core.pyx:5:0: Exception check on 'maximum_path_each' will always require the GIL to be acquired.
Possible solutions:
	1. Declare 'maximum_path_each' as 'noexcept' if you control the definition and you're sure you don't want the function to raise exceptions.
	2. Use an 'int' return type on 'maximum_path_each' to allow an error code to be returned.
performance hint: core.pyx:36:0: Exception check on 'maximum_path_c' will always require the GIL to be acquired.
Possible solutions:
	1. Declare 'maximum_path_c' as 'noexcept' if you control the definition and you're sure you don't want the function to raise exceptions.
	2. Use an 'int' return type on 'maximum_path_c' to allow an error code to be returned.
performance hint: core.pyx:42:21: Exception check after calling 'maximum_path_each' will always require the GIL to be acquired.
Possible solutions:
	1. Declare 'maximum_path_each' as 'noexcep

Compiling /content/piper/src/python/piper_train/vits/monotonic_align/core.pyx because it changed.
[1/1] Cythonizing /content/piper/src/python/piper_train/vits/monotonic_align/core.pyx
Piper ready


In [7]:
# === Auto-verify wake word IPA via espeak-ng (now installed by the previous cell) ===
if MODE == 'generate':
    import subprocess
    def espeak_ipa(text, voice):
        try:
            r = subprocess.run(['espeak-ng', '-v', voice, '-q', '--ipa', text],
                                capture_output=True, text=True, check=True)
            ipa = r.stdout.strip()
            return ipa if ipa else None
        except Exception as e:
            print(f'espeak-ng failed for {text!r} ({voice}): {e}')
            return None

    _us = espeak_ipa(WAKE_WORD, 'en-us')
    if _us:
        print(f'espeak-ng US IPA for {WAKE_WORD!r}: {_us!r} (was using fallback {WAKE_WORD_IPA_US!r})')
        WAKE_WORD_IPA_US = _us
    else:
        print(f'Falling back to manually-set WAKE_WORD_IPA_US={WAKE_WORD_IPA_US!r}')

    if WAKE_WORD_IPA_UK:
        _uk = espeak_ipa(WAKE_WORD, 'en-gb')
        if _uk:
            print(f'espeak-ng UK IPA for {WAKE_WORD!r}: {_uk!r} (was using fallback {WAKE_WORD_IPA_UK!r})')
            WAKE_WORD_IPA_UK = _uk
        else:
            print(f'Falling back to manually-set WAKE_WORD_IPA_UK={WAKE_WORD_IPA_UK!r}')

    print(f'Final IPA -> US: {WAKE_WORD_IPA_US!r}  UK: {WAKE_WORD_IPA_UK!r}')
else:
    print('Skipped (MODE != generate)')


espeak-ng US IPA for 'Okay PRAT': 'oʊkˈeɪ pɹˈæt' (was using fallback 'oʊkˈeɪ prˈæt')
espeak-ng UK IPA for 'Okay PRAT': 'əʊkˈeɪ pɹˈat' (was using fallback 'əʊkˈeɪ prˈæt')
Final IPA -> US: 'oʊkˈeɪ pɹˈæt'  UK: 'əʊkˈeɪ pɹˈat'


In [8]:
# === Generate positive samples (US English, optionally UK) ===
if MODE == 'generate':
    import os, subprocess, sys
    PIPER_SAMPLE_GENERATOR_DIR = '/content/piper-sample-generator'
    MODEL_PATH = 'models/en_US-libritts_r-medium.pt'
    PIPER_BATCH = 256  # drop to 128 on T4 if CUDA OOM

    def run_piper(target_word, max_samples, output_dir):
        cmd = [sys.executable, '-m', 'piper_sample_generator',
               target_word, '--phoneme-input', '--model', MODEL_PATH,
               '--max-samples', str(max_samples),
               '--batch-size', str(PIPER_BATCH),
               '--noise-scales', '0.5', '--noise-scale-ws', '0.6',
               '--output-dir', output_dir]
        env = os.environ.copy()
        env['PYTHONPATH'] = PIPER_PYTHON_DIR + ':' + env.get('PYTHONPATH', '')
        subprocess.run(cmd, check=True, env=env)

    os.makedirs('generated_samples', exist_ok=True)

    print(f'Generating {SAMPLES_US} US samples for {WAKE_WORD_IPA_US!r}...')
    run_piper(WAKE_WORD_IPA_US, SAMPLES_US, 'generated_samples')

    if WAKE_WORD_IPA_UK and SAMPLES_UK > 0:
        print(f'Generating {SAMPLES_UK} UK samples for {WAKE_WORD_IPA_UK!r}...')
        run_piper(WAKE_WORD_IPA_UK, SAMPLES_UK, 'generated_samples')

    n = sum(1 for f in os.listdir('generated_samples') if f.endswith('.wav'))
    print(f'Total positive samples: {n}')
else:
    print('Skipped (MODE != generate)')


Generating 30000 US samples for 'oʊkˈeɪ pɹˈæt'...


DEBUG:__main__:Loading models/en_US-libritts_r-medium.pt
INFO:__main__:Successfully loaded the model
DEBUG:__main__:CUDA available, using GPU
DEBUG:__main__:Batch 1/117 complete
DEBUG:__main__:Batch 2/117 complete
DEBUG:__main__:Batch 3/117 complete
DEBUG:__main__:Batch 4/117 complete
DEBUG:__main__:Batch 5/117 complete
DEBUG:__main__:Batch 6/117 complete
DEBUG:__main__:Batch 7/117 complete
DEBUG:__main__:Batch 8/117 complete
DEBUG:__main__:Batch 9/117 complete
DEBUG:__main__:Batch 10/117 complete
DEBUG:__main__:Batch 11/117 complete
DEBUG:__main__:Batch 12/117 complete
DEBUG:__main__:Batch 13/117 complete
DEBUG:__main__:Batch 14/117 complete
DEBUG:__main__:Batch 15/117 complete
DEBUG:__main__:Batch 16/117 complete
DEBUG:__main__:Batch 17/117 complete
DEBUG:__main__:Batch 18/117 complete
DEBUG:__main__:Batch 19/117 complete
DEBUG:__main__:Batch 20/117 complete
DEBUG:__main__:Batch 21/117 complete
DEBUG:__main__:Batch 22/117 complete
DEBUG:__main__:Batch 23/117 complete
DEBUG:__main__:B

Generating 15000 UK samples for 'əʊkˈeɪ pɹˈat'...


DEBUG:__main__:Loading models/en_US-libritts_r-medium.pt
INFO:__main__:Successfully loaded the model
DEBUG:__main__:CUDA available, using GPU
DEBUG:__main__:Batch 1/58 complete
DEBUG:__main__:Batch 2/58 complete
DEBUG:__main__:Batch 3/58 complete
DEBUG:__main__:Batch 4/58 complete
DEBUG:__main__:Batch 5/58 complete
DEBUG:__main__:Batch 6/58 complete
DEBUG:__main__:Batch 7/58 complete
DEBUG:__main__:Batch 8/58 complete
DEBUG:__main__:Batch 9/58 complete
DEBUG:__main__:Batch 10/58 complete
DEBUG:__main__:Batch 11/58 complete
DEBUG:__main__:Batch 12/58 complete
DEBUG:__main__:Batch 13/58 complete
DEBUG:__main__:Batch 14/58 complete
DEBUG:__main__:Batch 15/58 complete
DEBUG:__main__:Batch 16/58 complete
DEBUG:__main__:Batch 17/58 complete
DEBUG:__main__:Batch 18/58 complete
DEBUG:__main__:Batch 19/58 complete
DEBUG:__main__:Batch 20/58 complete
DEBUG:__main__:Batch 21/58 complete
DEBUG:__main__:Batch 22/58 complete
DEBUG:__main__:Batch 23/58 complete
DEBUG:__main__:Batch 24/58 complete
DEB

Total positive samples: 30000


In [9]:
# === Generate confusable negatives ===
if MODE == 'generate':
    import os, subprocess, sys
    from pathlib import Path
    PIPER_SAMPLE_GENERATOR_DIR = '/content/piper-sample-generator'
    MODEL_PATH = 'models/en_US-libritts_r-medium.pt'

    os.makedirs('confusable_negatives', exist_ok=True)
    for phrase in CONFUSABLE_PHRASES:
        safe = phrase.replace(' ', '_').replace(',', '').replace("'", '')
        existing = len(list(Path('confusable_negatives').glob(f'{safe}_*.wav')))
        if existing >= SAMPLES_PER_CONFUSABLE:
            print(f'  {phrase!r}: {existing} already, skip')
            continue
        tmp = f'/tmp/confusable_{safe}'
        os.makedirs(tmp, exist_ok=True)
        print(f'  generating {SAMPLES_PER_CONFUSABLE} for {phrase!r}...')
        _env = os.environ.copy()
        _env['PYTHONPATH'] = PIPER_PYTHON_DIR + ':' + _env.get('PYTHONPATH', '')
        subprocess.run([sys.executable, '-m', 'piper_sample_generator',
                        phrase, '--model', MODEL_PATH,
                        '--max-samples', str(SAMPLES_PER_CONFUSABLE),
                        '--batch-size', '256',
                        '--noise-scales', '0.5', '--noise-scale-ws', '0.6',
                        '--output-dir', tmp], check=True, env=_env)
        # Move + rename with safe prefix
        for f in os.listdir(tmp):
            if f.endswith('.wav'):
                os.rename(f'{tmp}/{f}', f'confusable_negatives/{safe}_{f}')

    n = sum(1 for f in os.listdir('confusable_negatives') if f.endswith('.wav'))
    print(f'Total confusable negatives: {n}')
else:
    print('Skipped (MODE != generate)')


  generating 1000 for 'okay then'...


DEBUG:__main__:Loading models/en_US-libritts_r-medium.pt
INFO:__main__:Successfully loaded the model
DEBUG:__main__:CUDA available, using GPU
DEBUG:__main__:Batch 1/3 complete
DEBUG:__main__:Batch 2/3 complete
DEBUG:__main__:Batch 3/3 complete
DEBUG:__main__:Batch 4/3 complete
INFO:__main__:Done


  generating 1000 for 'okay now'...


DEBUG:__main__:Loading models/en_US-libritts_r-medium.pt
INFO:__main__:Successfully loaded the model
DEBUG:__main__:CUDA available, using GPU
DEBUG:__main__:Batch 1/3 complete
DEBUG:__main__:Batch 2/3 complete
DEBUG:__main__:Batch 3/3 complete
DEBUG:__main__:Batch 4/3 complete
INFO:__main__:Done


  generating 1000 for 'okay so'...


DEBUG:__main__:Loading models/en_US-libritts_r-medium.pt
INFO:__main__:Successfully loaded the model
DEBUG:__main__:CUDA available, using GPU
DEBUG:__main__:Batch 1/3 complete
DEBUG:__main__:Batch 2/3 complete
DEBUG:__main__:Batch 3/3 complete
DEBUG:__main__:Batch 4/3 complete
INFO:__main__:Done


... (31 outputs left out)

DEBUG:__main__:Loading models/en_US-libritts_r-medium.pt
INFO:__main__:Successfully loaded the model
DEBUG:__main__:CUDA available, using GPU
DEBUG:__main__:Batch 1/3 complete
DEBUG:__main__:Batch 2/3 complete
DEBUG:__main__:Batch 3/3 complete
DEBUG:__main__:Batch 4/3 complete
INFO:__main__:Done


  generating 1000 for 'prattle'...


DEBUG:__main__:Loading models/en_US-libritts_r-medium.pt
INFO:__main__:Successfully loaded the model
DEBUG:__main__:CUDA available, using GPU
DEBUG:__main__:Batch 1/3 complete
DEBUG:__main__:Batch 2/3 complete
DEBUG:__main__:Batch 3/3 complete
DEBUG:__main__:Batch 4/3 complete
INFO:__main__:Done


Total confusable negatives: 20000


In [10]:
# === Download standard negative datasets (DNS challenge, AudioSet, MIT IRs) ===
# kahrendt/microwakeword on HuggingFace restructured from raw folder trees to
# .zip archives at the repo root (speech.zip, dinner_party.zip, no_speech.zip,
# dinner_party_eval.zip). The original snapshot_download(allow_patterns=[...])
# call here used folder-style glob patterns ('speech/*' etc) that no longer
# match anything in the new layout -- it would "succeed" while silently
# downloading zero files. Fixed to match kahrendt's own current working
# notebook (kahrendt/microWakeWord/notebooks/basic_training_notebook.ipynb):
# wget + unzip each archive directly.
import os, subprocess

NEG_DIR = 'negative_datasets'
os.makedirs(NEG_DIR, exist_ok=True)
_link_root = 'https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main/'
_filenames = ['dinner_party.zip', 'dinner_party_eval.zip', 'no_speech.zip', 'speech.zip']
for fname in _filenames:
    stem = fname[:-4]
    if os.path.exists(f'{NEG_DIR}/{stem}') and os.listdir(f'{NEG_DIR}/{stem}'):
        print(f'{stem}: cached, skipping')
        continue
    zip_path = f'{NEG_DIR}/{fname}'
    print(f'Downloading {fname} (this is large, can take a while)...')
    subprocess.run(['wget', '-q', '-O', zip_path, _link_root + fname], check=True)
    subprocess.run(['unzip', '-q', zip_path, '-d', NEG_DIR], check=True)
    os.remove(zip_path)
    print(f'  {stem}: ready')

# MIT room impulse responses for reverb augmentation
if not os.path.exists('mit_rirs') or not os.listdir('mit_rirs'):
    print('Downloading MIT room impulse responses...')
    subprocess.run(['mkdir', '-p', 'mit_rirs'], check=True)
    subprocess.run('cd mit_rirs && wget -q https://www.openslr.org/resources/28/rirs_noises.zip && unzip -q rirs_noises.zip',
                   shell=True, check=True)

# Background noise corpora (FMA + AudioSet 16 kHz subsets) for AddBackgroundNoise
# augmentation. The original fma_16k.tar / audioset_16k.tar hosted on
# kahrendt/microwakeword are gone (confirmed 404 -- that repo now only hosts
# the spectrogram-feature zips above). Recreating them properly means
# re-downloading raw AudioSet/FMA and resampling thousands of clips (see
# kahrendt's basic_training_notebook.ipynb cell 4), which is slow and adds a
# new failure surface. Treating as best-effort here: if unavailable, the
# model still trains with RIR reverb + EQ/pitch/distortion/color-noise
# augmentation, just without raw background-noise-bed diversity. This is a
# known quality gap for real-world ambient false-accept rate -- flagged in
# the final report, not silently swallowed.
BACKGROUND_NOISE_OK = []
for name, url in [
    ('fma_16k', 'https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main/fma_16k.tar'),
    ('audioset_16k', 'https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main/audioset_16k.tar'),
]:
    if os.path.exists(name) and os.listdir(name):
        BACKGROUND_NOISE_OK.append(name)
        continue
    print(f'Attempting {name} background corpus (best-effort)...')
    r = subprocess.run(f'mkdir -p {name} && cd {name} && wget -q {url} -O archive.tar && tar -xf archive.tar && rm archive.tar',
                        shell=True)
    if r.returncode == 0 and os.listdir(name):
        BACKGROUND_NOISE_OK.append(name)
        print(f'  {name}: ready')
    else:
        print(f'  {name}: unavailable (source URL gone), skipping -- will train without it')
        subprocess.run(['rm', '-rf', name])

print('All negative datasets ready')
print(f'Background noise corpora available for augmentation: {BACKGROUND_NOISE_OK or "NONE (RIR reverb + other augmentations still apply)"}')


  dinner_party: ready
  dinner_party_eval: ready
  no_speech: ready
  speech: ready
Attempting fma_16k background corpus (best-effort)...
  fma_16k: unavailable (source URL gone), skipping -- will train without it
Attempting audioset_16k background corpus (best-effort)...
  audioset_16k: unavailable (source URL gone), skipping -- will train without it
All negative datasets ready
Background noise corpora available for augmentation: NONE (RIR reverb + other augmentations still apply)


In [11]:
# === Augmentation + feature extraction ===
from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration
import os, shutil, traceback
from mmap_ninja.ragged import RaggedMmap

clips = Clips(
    input_directory='generated_samples',
    file_pattern='*.wav',
    max_clip_duration_s=None,
    remove_silence=True,
    random_split_seed=42,
    split_count=0.1,
)
augmenter = Augmentation(
    augmentation_duration_s=3.2,
    augmentation_probabilities={
        'SevenBandParametricEQ': 0.15, 'TanhDistortion': 0.10,
        'PitchShift': 0.15, 'BandStopFilter': 0.10,
        'AddColorNoise': 0.20, 'AddBackgroundNoise': 0.85,
        'Gain': 1.00, 'GainTransition': 0.25, 'RIR': 0.60,
    },
    impulse_paths=['mit_rirs'],
    background_paths=BACKGROUND_NOISE_OK,  # [] is handled natively by Augmentation -- background augmentation just skipped
    background_min_snr_db=-5, background_max_snr_db=20,
    min_jitter_s=0.10, max_jitter_s=0.50,
)

os.makedirs('generated_augmented_features', exist_ok=True)
SPLIT_CONFIG = {
    'training':   {'split_name': 'train',      'repetition': 3, 'slide_frames': 10},
    'validation': {'split_name': 'validation', 'repetition': 1, 'slide_frames': 10},
    'testing':    {'split_name': 'test',       'repetition': 1, 'slide_frames': 1 },
}

for split, cfg in SPLIT_CONFIG.items():
    out = f'generated_augmented_features/{split}'
    mmap = f'{out}/wakeword_mmap'
    if os.path.exists(mmap) and list(os.scandir(mmap)):
        print(f'{split}: cached, skipping')
        continue
    if os.path.exists(mmap):
        shutil.rmtree(mmap)
    os.makedirs(out, exist_ok=True)
    print(f'Generating {split} (rep={cfg["repetition"]}, slide={cfg["slide_frames"]})...')
    try:
        sg = SpectrogramGeneration(clips=clips, augmenter=augmenter,
                                    slide_frames=cfg['slide_frames'], step_ms=10)
        RaggedMmap.from_generator(
            out_dir=mmap, batch_size=200, verbose=True,
            sample_generator=sg.spectrogram_generator(
                split=cfg['split_name'], repeat=cfg['repetition']),
        )
    except Exception:
        traceback.print_exc()
        if os.path.exists(mmap): shutil.rmtree(mmap)
        raise
print('Positive features ready')

# Confusable features (only if confusable_negatives/ exists)
if os.path.exists('confusable_negatives') and os.listdir('confusable_negatives'):
    print('Generating confusable features...')
    confusable_clips = Clips(
        input_directory='confusable_negatives', file_pattern='*.wav',
        max_clip_duration_s=None, remove_silence=True,
        random_split_seed=42, split_count=0.1,
    )
    os.makedirs('confusable_features', exist_ok=True)
    for split, cfg in SPLIT_CONFIG.items():
        out = f'confusable_features/{split}'
        mmap = f'{out}/wakeword_mmap'
        if os.path.exists(mmap) and list(os.scandir(mmap)):
            continue
        if os.path.exists(mmap): shutil.rmtree(mmap)
        os.makedirs(out, exist_ok=True)
        sg = SpectrogramGeneration(clips=confusable_clips, augmenter=augmenter,
                                    slide_frames=cfg['slide_frames'], step_ms=10)
        RaggedMmap.from_generator(
            out_dir=mmap, batch_size=200, verbose=True,
            sample_generator=sg.spectrogram_generator(
                split=cfg['split_name'], repeat=cfg['repetition']),
        )
    print('Confusable features ready')

# Real recording features (only if real_recordings/ exists)
if os.path.exists('real_recordings') and os.listdir('real_recordings'):
    print('Generating real-recording features...')
    real_clips = Clips(
        input_directory='real_recordings', file_pattern='*.wav',
        max_clip_duration_s=None, remove_silence=True,
        random_split_seed=42, split_count=0.1,
    )
    os.makedirs('real_recording_features', exist_ok=True)
    for split, cfg in SPLIT_CONFIG.items():
        out = f'real_recording_features/{split}'
        mmap = f'{out}/wakeword_mmap'
        if os.path.exists(mmap) and list(os.scandir(mmap)):
            continue
        if os.path.exists(mmap): shutil.rmtree(mmap)
        os.makedirs(out, exist_ok=True)
        sg = SpectrogramGeneration(clips=real_clips, augmenter=augmenter,
                                    slide_frames=cfg['slide_frames'], step_ms=10)
        RaggedMmap.from_generator(
            out_dir=mmap, batch_size=200, verbose=True,
            sample_generator=sg.spectrogram_generator(
                split=cfg['split_name'], repeat=cfg['repetition']),
        )
    print('Real-recording features ready')


Generating training (rep=3, slide=10)...


0it [00:00, ?it/s]

Generating validation (rep=1, slide=10)...


0it [00:00, ?it/s]

Generating testing (rep=1, slide=1)...


0it [00:00, ?it/s]

Positive features ready
Generating confusable features...


0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

Confusable features ready


In [12]:
# === Training config YAML ===
import yaml, os
from pathlib import Path

SKIP_CONFUSABLES = not Path('confusable_features/training/wakeword_mmap').exists()
SKIP_REAL = not Path('real_recording_features/training/wakeword_mmap').exists()

config = {
    'window_step_ms': 10,
    'train_dir': f'trained_models/{OUTPUT_NAME}',
    'features': [
        dict(features_dir='generated_augmented_features', sampling_weight=8.0,
             penalty_weight=2.0, truth=True, truncation_strategy='truncate_start',
             type='mmap'),
        dict(features_dir='negative_datasets/speech', sampling_weight=10.0,
             penalty_weight=2.5, truth=False, truncation_strategy='random', type='mmap'),
        dict(features_dir='negative_datasets/dinner_party', sampling_weight=15.0,
             penalty_weight=3.0, truth=False, truncation_strategy='random', type='mmap'),
        dict(features_dir='negative_datasets/no_speech', sampling_weight=5.0,
             penalty_weight=1.0, truth=False, truncation_strategy='random', type='mmap'),
        dict(features_dir='negative_datasets/dinner_party_eval', sampling_weight=0.0,
             penalty_weight=1.0, truth=False, truncation_strategy='split', type='mmap'),
    ],
    'training_steps': [25000, 20000],
    'positive_class_weight': [2, 2],
    'negative_class_weight': [40, 50],
    'learning_rates': [0.001, 0.0001],
    'batch_size': 256,
    'time_mask_max_size': [5, 5], 'time_mask_count': [1, 1],
    'freq_mask_max_size': [3, 3], 'freq_mask_count': [1, 1],
    'eval_step_interval': 500,
    'clip_duration_ms': 1500,
    'target_minimization': 0.4,
    'minimization_metric': 'ambient_false_positives_per_hour',
    'maximization_metric': 'average_viable_recall',
}

if not SKIP_CONFUSABLES:
    config['features'].append(dict(features_dir='confusable_features', sampling_weight=8.0,
                                    penalty_weight=5.0, truth=False,
                                    truncation_strategy='random', type='mmap'))
if not SKIP_REAL:
    config['features'].append(dict(features_dir='real_recording_features', sampling_weight=8.0,
                                    penalty_weight=2.0, truth=True,
                                    truncation_strategy='truncate_start', type='mmap'))

os.makedirs(f'trained_models/{OUTPUT_NAME}', exist_ok=True)
with open('training_parameters.yaml', 'w') as f:
    yaml.dump(config, f)
print('training_parameters.yaml ready')
print(f'  Feature sets: {len(config["features"])}, total steps: {sum(config["training_steps"])}')


training_parameters.yaml ready
  Feature sets: 6, total steps: 45000


In [13]:
# === Train the model ===
# subprocess + PYTHONPATH so it can find microwakeword (sys.path doesn't propagate)
import os, sys, subprocess, shutil
shutil.rmtree(f'trained_models/{OUTPUT_NAME}', ignore_errors=True)

env = os.environ.copy()
env['PYTHONPATH'] = '/content/microWakeWord:' + env.get('PYTHONPATH', '')
env['XLA_FLAGS'] = '--xla_gpu_autotune_level=0'

# webrtcvad (a microWakeWord dep) transitively imports pkg_resources at
# module level. This is the ONLY pkg_resources on this container (confirmed:
# an earlier attempt moved it aside assuming a modern pip-installed copy
# would take over -- there wasn't one, and that broke everything with
# ModuleNotFoundError instead). Its legacy namespace-package declaration code
# (_handle_ns, run automatically at import time by pkg_resources itself)
# calls two APIs Python 3.12 removed: pkgutil.ImpImporter, and
# importlib.machinery.FileFinder.find_module (replaced by find_spec).
# sitecustomize.py on PYTHONPATH is auto-imported by Python's site module at
# interpreter startup, before webrtcvad gets a chance to import pkg_resources,
# so we shim both missing APIs back in (not delete/replace the module).
with open('/content/microWakeWord/sitecustomize.py', 'w') as _f:
    _f.write(
        'import pkgutil\n'
        'import importlib.machinery\n'
        '\n'
        'if not hasattr(pkgutil, "ImpImporter"):\n'
        '    class ImpImporter:\n'
        '        def __init__(self, *a, **kw): pass\n'
        '    pkgutil.ImpImporter = ImpImporter\n'
        '\n'
        'if not hasattr(importlib.machinery.FileFinder, "find_module"):\n'
        '    def _find_module(self, fullname, path=None):\n'
        '        spec = self.find_spec(fullname, path)\n'
        '        return spec.loader if spec else None\n'
        '    importlib.machinery.FileFinder.find_module = _find_module\n'
    )

cmd = [
    sys.executable, '-m', 'microwakeword.model_train_eval',
    '--training_config', 'training_parameters.yaml',
    '--train', '1', '--restore_checkpoint', '0',
    '--test_tflite_streaming_quantized', '1',
    '--use_weights', 'best_weights',
    'mixednet',
    '--pointwise_filters', '64,64,64,64',
    '--repeat_in_block', '1, 1, 1, 1',
    '--mixconv_kernel_sizes', '[5], [7,11], [9,15], [23]',
    '--residual_connection', '0,0,0,0',
    '--first_conv_filters', '32',
    '--first_conv_kernel_size', '5',
    '--stride', '3',
]
print('Running:', ' '.join(cmd)); print()
proc = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
proc.wait()
print(); print('Exit code:', proc.returncode)
assert proc.returncode == 0, 'training failed - see output above'


Running: /usr/bin/python3 -m microwakeword.model_train_eval --training_config training_parameters.yaml --train 1 --restore_checkpoint 0 --test_tflite_streaming_quantized 1 --use_weights best_weights mixednet --pointwise_filters 64,64,64,64 --repeat_in_block 1, 1, 1, 1 --mixconv_kernel_sizes [5], [7,11], [9,15], [23] --residual_connection 0,0,0,0 --first_conv_filters 32 --first_conv_kernel_size 5 --stride 3

INFO:absl:Loading and analyzing data sets.
I0000 00:00:1790176160.012999     673 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1790176160.016291     673 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
Model: "functional"
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer

In [14]:
# === Export + push to Drive ===
import os, json, shutil, datetime

tflite_src = f'trained_models/{OUTPUT_NAME}/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite'
assert os.path.exists(tflite_src), f'No model at {tflite_src}'

OUT_TFLITE = f'{OUTPUT_NAME}.tflite'
OUT_JSON = f'{OUTPUT_NAME}.json'
shutil.copy2(tflite_src, OUT_TFLITE)
print(f'wrote {OUT_TFLITE} ({os.path.getsize(OUT_TFLITE)/1024:.1f} KB)')

manifest = {
    'type': 'micro',
    'wake_word': WAKE_WORD,
    'author': AUTHOR,
    'website': AUTHOR_WEBSITE,
    'model': OUT_TFLITE,
    'trained_languages': TRAINED_LANGUAGES,
    'version': 2,
    'micro': {
        'probability_cutoff': PROBABILITY_CUTOFF,
        'feature_step_size': 10,
        'sliding_window_size': SLIDING_WINDOW_SIZE,
        'tensor_arena_size': TENSOR_ARENA_SIZE,
        'minimum_esphome_version': '2024.7.0',
    }
}
with open(OUT_JSON, 'w') as f:
    json.dump(manifest, f, indent=2)
print(f'wrote {OUT_JSON}')
print(json.dumps(manifest, indent=2))

for fn in (OUT_TFLITE, OUT_JSON):
    shutil.copy2(fn, f'{DRIVE_DIR}/{fn}')
    print(f'pushed {fn} -> {DRIVE_DIR}')

ts = datetime.datetime.now(datetime.timezone.utc).isoformat()
with open(f'{DRIVE_DIR}/_run_finished.txt', 'w') as f:
    f.write(f'Training run finished at {ts}\n')
print()
print(f'DONE. Find your model at {DRIVE_DIR}/{OUTPUT_NAME}.tflite')


wrote okay_prat.tflite (59.5 KB)
wrote okay_prat.json
{
  "type": "micro",
  "wake_word": "Okay PRAT",
  "author": "prat",
  "website": "https://github.com/prat",
  "model": "okay_prat.tflite",
  "trained_languages": [
    "en"
  ],
  "version": 2,
  "micro": {
    "probability_cutoff": 0.85,
    "feature_step_size": 10,
    "sliding_window_size": 5,
    "tensor_arena_size": 50000,
    "minimum_esphome_version": "2024.7.0"
  }
}
pushed okay_prat.tflite -> /kaggle/working/wakeword_training_okay_prat
pushed okay_prat.json -> /kaggle/working/wakeword_training_okay_prat

DONE. Find your model at /kaggle/working/wakeword_training_okay_prat/okay_prat.tflite


## Deploying to ESPHome devices

Drop the `.tflite` + `.json` next to your ESPHome YAML (e.g. in `/config/esphome/wakewords/<output_name>/`).

In your device YAML, replace your existing wake word:

```yaml
micro_wake_word:
  models:
    - model: wakewords/<output_name>/<output_name>.json
  on_wake_word_detected:
    - voice_assistant.start:
```

Then `esphome run <device>.yaml` (USB or OTA).

## Manifest tuning (likely needed)

The defaults work for **Hey Harold** specifically. Your model's confidence
distribution will differ. Iterate:

| Symptom | Knob |
|---|---|
| Doesn't fire on the wake word | Lower `probability_cutoff` (try 0.7, 0.6) |
| Fires on too many things | Raise `probability_cutoff` (try 0.92, 0.95) |
| LED fires but no STT response | `sliding_window_size: 5` (faster fire); also check Echo speaker mute |
| `Failed to allocate tensors` log | Raise `tensor_arena_size` (try 50000, 80000) |

## Known gotchas
- Editable install (`pip install -e ./microWakeWord`) requires kernel restart — broken for Run All
- `train.py` upstream calls `.numpy()` on numpy arrays under newer TF (this notebook patches it)
- T4 GPU OOMs during validation — use A100 + High-RAM
- Manifest path mismatch: training writes to `trained_models/<output_name>/`, manifest must match
- `voice_assistant` has no audio lookback; if MWW fires AFTER user speaks, STT gets silence
